In [13]:
import os, random, shutil

def split_data(src, dst, train=700, val=150):
    imgs = os.listdir(src)
    random.shuffle(imgs)

    splits = {
        "train": imgs[:train],
        "val": imgs[train:train+val],
        "test": imgs[train+val:]
    }

    for split, files in splits.items():
        os.makedirs(dst.replace("SPLIT", split), exist_ok=True)
        for f in files:
            shutil.copy(os.path.join(src, f), dst.replace("SPLIT", split))

split_data(
    "dataset/Normal/images",
    "data_split/SPLIT/Normal"
)

split_data(
    "dataset/Pneumonia/images",
    "data_split/SPLIT/Pneumonia"
)


In [9]:
import os
from PIL import Image
from torch.utils.data import Dataset
import torch

class PneumoniaDataset(Dataset):
    def __init__(self, root, transform=None):
        self.data = []
        self.transform = transform

        for label, cls in enumerate(["Normal", "Pneumonia"]):
            folder = os.path.join(root, cls)
            for img in os.listdir(folder):
                self.data.append((os.path.join(folder, img), label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_data = datasets.ImageFolder("data_split/train", transform=train_transform)
val_data   = datasets.ImageFolder("data_split/val", transform=val_transform)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_data, batch_size=16, shuffle=False, num_workers=0)

model = models.mobilenet_v2(pretrained=True)
model.classifier = nn.Sequential(
    nn.Linear(model.last_channel, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 1),
    nn.Sigmoid()
)

model.to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    print(f"\nEpoch {epoch+1}/{epochs} started...")

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 10 == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Batch [{batch_idx}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} finished | Avg Loss: {epoch_loss:.4f}")
    
torch.save(model.state_dict(), "mobilenetv2_pneumonia.pth")
print("Model saved.")



Epoch 1/5 started...
Epoch [1/5] Batch [0/116] Loss: 0.7014
Epoch [1/5] Batch [10/116] Loss: 0.4411
Epoch [1/5] Batch [20/116] Loss: 0.2902
Epoch [1/5] Batch [30/116] Loss: 0.1633
Epoch [1/5] Batch [40/116] Loss: 0.1019
Epoch [1/5] Batch [50/116] Loss: 0.1569
Epoch [1/5] Batch [60/116] Loss: 0.3849
Epoch [1/5] Batch [70/116] Loss: 0.0273
Epoch [1/5] Batch [80/116] Loss: 0.0687
Epoch [1/5] Batch [90/116] Loss: 0.2473
Epoch [1/5] Batch [100/116] Loss: 0.0441
Epoch [1/5] Batch [110/116] Loss: 0.0409
Epoch 1 finished | Avg Loss: 0.2180

Epoch 2/5 started...
Epoch [2/5] Batch [0/116] Loss: 0.0753
Epoch [2/5] Batch [10/116] Loss: 0.7235
Epoch [2/5] Batch [20/116] Loss: 0.0426
Epoch [2/5] Batch [30/116] Loss: 0.0361
Epoch [2/5] Batch [40/116] Loss: 0.2350
Epoch [2/5] Batch [50/116] Loss: 0.0167
Epoch [2/5] Batch [60/116] Loss: 0.0173
Epoch [2/5] Batch [70/116] Loss: 0.1279
Epoch [2/5] Batch [80/116] Loss: 0.0137
Epoch [2/5] Batch [90/116] Loss: 0.4217
Epoch [2/5] Batch [100/116] Loss: 0.0111

In [3]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score,roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder("data_split/test", transform=transform)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

model = models.mobilenet_v2(pretrained=False)
model.classifier = nn.Sequential(
    nn.Linear(model.last_channel, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 1),
    nn.Sigmoid()
)

model.load_state_dict(torch.load("mobilenetv2_pneumonia.pth", map_location=device))
model.to(device)
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).cpu().numpy()
        preds = (outputs > 0.5).astype(int)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, y_pred))



c:\Users\Admin\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Accuracy: 0.9965811965811966
F1: 0.9966887417218543
ROC-AUC: 0.9965775864086304
